# Deploying with LangGraph Platform

Once you've built a LangGraph agent, **LangGraph Platform** is the official way to ship it to production. It gives you:

- A managed runtime for your graphs
- Built-in persistence, threads, and assistants APIs
- LangGraph Studio (visual debugging UI)
- Multiple deployment models — from fully managed cloud to self-hosted

## Project Structure

A LangGraph Platform project follows a standard layout. The `langgraph.json` manifest at the root tells the platform how to find and run your graphs.

```
my-agent/
├── src/
│   └── my_agent/
│       ├── __init__.py
│       ├── graph.py          # Your StateGraph definition
│       ├── state.py          # State TypedDicts
│       ├── configuration.py  # Configurable fields
│       ├── prompts.py        # System prompts
│       └── utils.py          # Helper functions
├── langgraph.json            # LangGraph Platform config
├── requirements.txt
└── .env
```

Keep your graph logic in `graph.py` and export a top-level `graph` variable — the platform looks it up by name.

## langgraph.json

The `langgraph.json` file is the **deployment manifest**. It tells the platform:

- Where your dependencies live (`dependencies`)
- Which graphs to expose and at what import path (`graphs`)
- Where to load environment variables from (`env`)

```json
{
  "dependencies": ["."],
  "graphs": {
    "agent": "./src/my_agent/graph.py:graph"
  },
  "env": ".env"
}
```

The key on the right of the colon (`graph` in `./src/my_agent/graph.py:graph`) must be a top-level variable in that module — typically a compiled `StateGraph`.

## Local Development

Before deploying, you can run your project locally with the LangGraph CLI. The `langgraph dev` command spins up a local server that mirrors the production runtime.

- Runs on **port 2024** by default
- **Hot-reloads** on code changes
- Includes **LangGraph Studio UI** at `http://localhost:2024`

Install the CLI and start the dev server:

```bash
pip install langgraph-cli
langgraph dev
```

Open `http://localhost:2024` in your browser to inspect threads, trace runs, and step through state.

## Deploying to LangSmith

When you're ready to ship, the simplest deployment target is **LangSmith Deployments** — the managed Cloud SaaS option run by LangChain.

```bash
langgraph deploy
```

This command:

1. Reads your `langgraph.json`
2. Builds a container image for your project
3. Pushes it to the LangSmith Deployments service
4. Returns a hosted endpoint you can call from any client

LangSmith manages scaling, persistence (Postgres), and observability for you.

## Deployment Options

LangGraph Platform supports four deployment models. Pick the one that matches your data-residency, cost, and control requirements:

| Option | Description | Cost |
|--------|-------------|------|
| Cloud SaaS | Hosted by LangChain on their infrastructure | Paid |
| Self-Hosted Lite | Run on your own infra, free tier available | Free (limited) |
| BYOC | Bring Your Own Cloud — your infra, LangChain manages | Paid |
| Self-Hosted Enterprise | Full control, enterprise support | Enterprise pricing |

## Minimal `graph.py` Example

Here is the smallest possible graph you can deploy. In a real project, the `chat` node would call an LLM and your graph would have routing logic, tools, and state.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class State(TypedDict):
    messages: list

def chat(state: State):
    # In a real app, call an LLM here
    return {"messages": state["messages"] + [{"role": "assistant", "content": "Hello!"}]}

builder = StateGraph(State)
builder.add_node("chat", chat)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)

graph = builder.compile()

## Further Reading

- **LangGraph Platform overview** — https://langchain-ai.github.io/langgraph/concepts/langgraph_platform/
- **Deployment options** — https://langchain-ai.github.io/langgraph/concepts/deployment_options/
- **LangGraph CLI reference** — https://langchain-ai.github.io/langgraph/cloud/reference/cli/
- **`langgraph.json` reference** — https://langchain-ai.github.io/langgraph/cloud/reference/cli/#configuration-file
- **LangGraph Studio** — https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/